# Part teòrica

**Variables:** Files i columnes (caselles del taulell)

**Domini:** Paraules del diccionari

**Restriccions:**
- Files i columnes han d'estar dins del taulell
- Files i columnes > 1 casella
- Interseccions de files i columnes tenen la mateixa lletra
- Si es troba una # acaba la fila o la columna
- Les paraules han d'estar escrites de dalt a baix i d'esquerra a dreta
- No es pot repetir una paraula
- Les paraules han de tenir una llargada menor o igual al màxim de m o n del taulell

**Tamany espai de solucions inicial:** k^(mxn) on m son les files, n les columnes del taulell y k les lletres de l'alfabet

**Estratègia de millora:** Fer ús d'una heurística per a decidir quina assignació fer a continuació. Per exemple triar una paraula amb el major nombre de restriccions respecte a altres no asignades, ja que si té un major impacte sobre la resta pot portar a acabar abans


# Codi

## Carregar biblioteques

In [177]:
import numpy as np

## Exercici 1

In [178]:
data = []
words = []

with open('crossword_CB_v3.txt', 'r') as file:
    for line in file.readlines():
        elements = line.strip().split()
        data.append(elements)
        
taulell = np.array(data)

with open('diccionari_CB_v3.txt', 'r') as file:
    for word in file.readlines():
        words.append(word.strip())

diccionary = np.array(words)

print(taulell)


[['0' '0' '0' '0' '0' '0']
 ['0' '#' '#' '0' '#' '0']
 ['0' '#' '0' '0' '0' '0']
 ['0' '#' '#' '0' '#' '0']
 ['#' '0' '0' '0' '0' '0']
 ['0' '0' '0' '0' '#' '#']
 ['0' '0' '#' '#' '#' '#']]


In [179]:
#variable = [[[pos_inicial], len, v/h]] vertical = 1, horitzontal = 0

def cercaVariablesHoritzontal(taulell, variables):
    for n, i in enumerate(taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [180]:
def cercaVariablesVertical(taulell, variables):
    transposed_taulell = taulell.transpose()
    for n, i in enumerate(transposed_taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [181]:
def cercaVariables(taulell, variables):
    cercaVariablesHoritzontal(taulell, variables)
    cercaVariablesVertical(taulell, variables)

In [182]:
def calculaPosicionsVariable(variable):
    variable_positions = []
    for x in range(variable[1]):
        i, j = 0, 0
        if variable[2] == 0: i = x
        else: j = x
        variable_positions.append([variable[0][0]+j, variable[0][1]+i])
    
    return variable_positions

In [183]:
def interseccions(variables):
    positions = []
    intersections = []
    for variable in variables:
        for pos in calculaPosicionsVariable(variable):
            if pos not in positions:
                positions.append(pos)
            elif pos not in intersections:
                intersections.append(pos)
                
    
    return(intersections)

In [184]:
def isValid(word, variable, variables, assigned_variables, taulell):
    if len(word) != variable[1]: return False
    for assigned_variable in assigned_variables:
        if word == assigned_variable[3]: return False
    
    for position in calculaPosicionsVariable(variable):
        if position in interseccions(variables) and taulell[position[0]][position[1]] != '0':
            if variable[2] == 0:
                pos_interseccio = position[1] - variable[0][1]
                if word[pos_interseccio] != taulell[position[0]][position[1]]: return False
            elif variable[2] == 1:
                pos_interseccio = position[0] - variable[0][0]
                if word[pos_interseccio] != taulell[position[0]][position[1]]: return False
        
    return True

In [185]:
def backtracking(assigned_variables, variables, taulell):
    with open('diccionari_CB_v3.txt', 'r') as dic:
        words = [line.strip() for line in dic]
        for word in words:
            for variable in variables:
                if isValid(word, variable, variables, assigned_variables, taulell):
                    variable.append(word)
                    assigned_variables.append(variable)
                    variables.remove(variable)

assigned_variables = []
variables = []
cercaVariables(taulell, variables)

backtracking(assigned_variables, variables, taulell)

print(assigned_variables)

[[[0, 0], 6, 0, 'ACATAR'], [[2, 2], 4, 0, 'ALTA'], [[5, 0], 4, 0, 'BORE'], [[0, 0], 4, 1, 'CARA'], [[0, 3], 6, 1, 'CARNET'], [[4, 1], 3, 1, 'COR'], [[4, 1], 5, 0, 'DIARI'], [[6, 0], 2, 0, 'DO'], [[0, 5], 5, 1, 'DORAT'], [[5, 0], 2, 1, 'LA'], [[4, 2], 2, 1, 'MI']]


In [186]:
def printTaulell(assigned_variables, taulell):
    for assigned_variable in assigned_variables:
        for position in calculaPosicionsVariable(assigned_variable):
            if assigned_variable[2] == 0:
                dist = assigned_variable[0][1] - position[1]
                taulell[position[0]][position[1]] = assigned_variable[3][dist]
            elif assigned_variable[2] == 1:
                dist = assigned_variable[0][0] - position[0]
                taulell[position[0]][position[1]] = assigned_variable[3][dist]
                
printTaulell(assigned_variables, taulell)
taulell

array([['C', 'R', 'A', 'C', 'A', 'D'],
       ['A', '#', '#', 'T', '#', 'T'],
       ['R', '#', 'A', 'E', 'T', 'A'],
       ['A', '#', '#', 'N', '#', 'R'],
       ['#', 'D', 'M', 'R', 'A', 'O'],
       ['L', 'R', 'I', 'A', '#', '#'],
       ['A', 'O', '#', '#', '#', '#']], dtype='<U1')